# VoyageAI - Destination Agent

This notebook creates the Destination Agent for VoyageAI.

The Destination Agent is responsible for:
1. Understanding the user's travel preference
2. Retrieving relevant travel knowledge from ChromaDB
3. Selecting the most suitable destination
4. Returning a structured recommendation

In [44]:
from pathlib import Path
from dotenv import load_dotenv
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [45]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [46]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "agents":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Current directory: e:\Agentic AI\VoyageAI Backend\notebooks\agents
Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [47]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7427.89it/s]


Embedding model loaded successfully.


In [48]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 45


In [49]:
def get_available_destinations():
    collection_data = vector_store._collection.get(include=["metadatas"])

    destinations = set()

    for metadata in collection_data["metadatas"]:
        city = metadata.get("city")
        if city:
            destinations.add(city.lower().strip())

    return sorted(list(destinations))


available_destinations = get_available_destinations()

print("Available destinations in ChromaDB:")
print(available_destinations)

Available destinations in ChromaDB:
['agra', 'andaman', 'darjeeling', 'delhi', 'goa', 'jaipur', 'kashmir', 'kerala', 'ladakh', 'manali', 'mumbai', 'pondicherry', 'rishikesh', 'udaipur', 'varanasi']


In [50]:
def normalize_text(text: str):
    if not text:
        return ""

    return text.lower().strip()


def is_destination_in_kb(destination_name: str):
    destination_name = normalize_text(destination_name)

    if not destination_name:
        return False

    for destination in available_destinations:
        if destination == destination_name:
            return True

        if destination in destination_name:
            return True

        if destination_name in destination:
            return True

    return False

In [51]:
def get_retrieval_context(query: str, k: int = 3):
    docs = vector_store.similarity_search(query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [60]:
user_query = "I want a romantic luxury trip with lakes and palaces"

context = get_retrieval_context(user_query, k=3)

print(context)

Context 1
City: udaipur
Source: udaipur.txt
Content:
Top Attractions:
- City Palace
- Lake Pichola
- Jag Mandir
- Fateh Sagar Lake
- Sajjangarh Monsoon Palace
- Saheliyon Ki Bari
- Bagore Ki Haveli
- Jagdish Temple

Food Recommendations:
- Dal baati churma
- Rajasthani thali
- Laal maas
- Gatte ki sabzi
- Kachori
- Rooftop cafe meals near lakes

Stay Suggestions:
- Budget: guesthouses near old city
- Comfort: lake-view boutique hotels
- Luxury: palace hotels and lake resorts

Best Time to Visit:
October to March is best for pleasant sightseeing.

Context 2
City: kashmir
Source: kashmir.txt
Content:
Top Attractions:
- Srinagar
- Dal Lake
- Gulmarg
- Pahalgam
- Sonamarg
- Mughal Gardens
- Shankaracharya Temple
- Betaab Valley

Food Recommendations:
- Rogan josh
- Yakhni
- Dum aloo
- Kahwa
- Wazwan
- Kashmiri pulao
- Sheer chai

Stay Suggestions:
- Budget: guesthouses in Srinagar
- Comfort: houseboats or hotels near Dal Lake
- Luxury: premium resorts in Gulmarg or Pahalgam

Context 3
City

In [53]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [54]:
destination_agent_prompt = ChatPromptTemplate.from_template("""
You are the Destination Agent of VoyageAI.

Your task is to recommend the best destination based only on the available VoyageAI knowledge base.

Available destinations in the knowledge base:
{available_destinations}

User Travel Request:
{user_query}

Retrieved Travel Knowledge:
{context}

Return your answer strictly in valid JSON format.

JSON structure:
{{
  "agent_name": "Destination Agent",
  "status": "success/out_of_knowledge_base",
  "recommended_destination": "destination name or null",
  "reason": "short reason",
  "suitable_for": ["type 1", "type 2", "type 3"],
  "suggested_duration": "duration from retrieved knowledge or null",
  "best_time_to_visit": "best time from retrieved knowledge or null",
  "key_attractions": ["attraction 1", "attraction 2", "attraction 3"],
  "confidence": "high/medium/low"
}}

Rules:
- You are allowed to recommend ONLY destinations from the available destinations list.
- Do not use your general knowledge.
- Do not invent destinations outside the available destinations list.
- If the user explicitly asks for a destination that is not in the available destinations list, set status to "out_of_knowledge_base".
- If status is "out_of_knowledge_base", set recommended_destination to null.
- If the retrieved knowledge does not support the answer, set confidence to "low".
- Do not add markdown.
- Do not wrap JSON in ```json.
""")

In [55]:
destination_agent_chain = destination_agent_prompt | llm | StrOutputParser()

print("Destination Agent chain created successfully.")

Destination Agent chain created successfully.


In [58]:
def run_destination_agent(user_query: str, k: int = 3):
    context = get_retrieval_context(user_query, k=k)

    response = destination_agent_chain.invoke({
        "user_query": user_query,
        "context": context,
        "available_destinations": ", ".join(available_destinations)
    })

    try:
        parsed_response = json.loads(response)
    except json.JSONDecodeError:
        print("Warning: LLM did not return valid JSON. Raw response below:")
        print(response)

        return {
            "agent_name": "Destination Agent",
            "status": "error",
            "recommended_destination": None,
            "reason": "Failed to parse agent response.",
            "suitable_for": [],
            "suggested_duration": None,
            "best_time_to_visit": None,
            "key_attractions": [],
            "confidence": "low",
            "raw_response": response
        }

    recommended_destination = parsed_response.get("recommended_destination")

    if recommended_destination and not is_destination_in_kb(recommended_destination):
        return {
            "agent_name": "Destination Agent",
            "status": "out_of_knowledge_base",
            "recommended_destination": None,
            "reason": f"{recommended_destination} is not available in the current VoyageAI knowledge base.",
            "suitable_for": [],
            "suggested_duration": None,
            "best_time_to_visit": None,
            "key_attractions": [],
            "available_destinations": available_destinations,
            "confidence": "low"
        }

    return parsed_response

In [61]:
result = run_destination_agent(
    "I want a romantic luxury trip with lakes and palaces"
)

print(json.dumps(result, indent=2))

{
  "agent_name": "Destination Agent",
  "status": "success",
  "recommended_destination": "udaipur",
  "reason": "palaces and lakes",
  "suitable_for": [
    "romantic",
    "luxury"
  ],
  "suggested_duration": "4-5 days",
  "best_time_to_visit": "October to March",
  "key_attractions": [
    "City Palace",
    "Lake Pichola",
    "Jag Mandir"
  ],
  "confidence": "high"
}


In [ ]:
test_queries = [
    "I want scuba diving, clean beaches and island hopping",
    "I want lakes, palaces and a romantic luxury trip",
    "I want yoga, rafting and a budget backpacking trip",
    "I want spiritual ghats, temples and evening aarti",
    "I want high altitude mountains, biking and monasteries",
    "I want French cafes, beaches and a weekend trip",
    "I want Taj Mahal, Mughal history and architecture"
]

for query in test_queries:
    print("=" * 100)
    print("User Query:", query)
    print("-" * 100)

    result = run_destination_agent(query)

    print(json.dumps(result, indent=2))
    print()

User Query: I want scuba diving, clean beaches and island hopping
----------------------------------------------------------------------------------------------------
{
  "agent_name": "Destination Agent",
  "status": "success",
  "recommended_destination": "Andaman and Nicobar Islands",
  "reason": "Best for scuba diving, clean beaches, and island hopping",
  "suitable_for": [
    "beach lovers",
    "honeymoon couples",
    "scuba diving beginners",
    "snorkeling lovers",
    "nature lovers",
    "luxury island travelers"
  ],
  "suggested_duration": "4-7 days",
  "best_time_to_visit": "October to March",
  "key_attractions": [
    "Radhanagar Beach",
    "Havelock Island",
    "Neil Island",
    "Cellular Jail"
  ],
  "confidence": "high"
}

User Query: I want lakes, palaces and a romantic luxury trip
----------------------------------------------------------------------------------------------------
{
  "agent_name": "Destination Agent",
  "status": "success",
  "recommended_dest